# 12. 팩터 귀속 — 이 전략이 새로운 것을 찾았는가

지금까지는 "안정군이 나머지군보다 위험대비수익이 높았다"까지 확인했다. 07번의 유의성 검정에서
그 우위가 통계적으로 유의하지 않다는 것도 확인했다. 여기서는 **팩터 리서치가 가장 먼저 묻는 질문**에
답한다.

> 저 우위(혹은 그 비슷한 것)가 **이미 알려진 팩터로 설명되고 남는 게 있는가?**

저베타·저변동성으로 종목을 고르면 기계적으로 대형주·방어주 쪽으로 기운다. 그건 Fama-French
팩터가 이미 가격을 매기고 있는 노출이다. 그래서 "위험대비수익이 높았다"는 것만으로는 아직
아무것도 발견하지 못한 것일 수 있다. 검증할 회귀식은 이것이다.

$$ r_p - r_f = \alpha + b_{mkt}(Mkt\!-\!RF) + b_{smb}SMB + b_{hml}HML + b_{mom}MOM + \epsilon $$

- **알파가 유의하게 양수**라면 → 네 팩터가 설명하지 못하는 무언가를 찾은 것
- **알파가 0과 구별되지 않는다**면 → 이 전략은 (나쁘지 않은) **알려진 팩터 노출을 사는 방법**일 뿐,
  새로운 수익원이 아니다

두 대상을 각각 회귀한다. **① 안정군 포트폴리오 자체**(무위험수익률 차감)로 노출 구조를 보고,
**② 롱숏 스프레드**(안정군 − 나머지군, 자기자금이 안 들어가므로 무위험수익률을 빼지 않음)로
전략의 "베팅" 자체에 알파가 있는지 본다.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/choedasom/lab_middle_project')
assert (PROJECT_ROOT / 'src' / 'factors.py').is_file(), f'경로 확인 필요: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

for m in list(sys.modules):
    if m == 'src' or m.startswith('src.'):
        del sys.modules[m]

import pandas as pd

from src.factors import load_ff_factors, compound_factors_to_periods, factor_regression
from src.performance import add_forward_returns, summarize_by_group

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
final_df = pd.read_parquet(PROCESSED_DIR / 'final_df.parquet')

ff = load_ff_factors()
print(f'팩터 기간: {ff.index.min().date()} ~ {ff.index.max().date()}')
ff.tail(3)

[캐시] Fama-French 팩터 26,195일 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/ff_factors_daily.csv)
팩터 기간: 1926-11-03 ~ 2026-07-31


,Mkt-RF,SMB,HML,RF,MOM
Date,,,,,
2026-07-29,-0.0152,0.0031,0.0031,0.0002,-0.0197
2026-07-30,0.0159,-0.0070,-0.0136,0.0002,0.0213
2026-07-31,0.0068,-0.0049,-0.0059,0.0002,-0.0086


## 팩터를 전략의 보유기간에 맞춰 복리 누적

일별 팩터 수익률을 그대로 쓰면 안 된다. 전략 수익률은 리밸런싱 시점 T+1부터 다음 시점 T+1까지의
구간 수익률이므로, **팩터도 정확히 같은 날짜 구간에서 복리로 누적**해야 좌변과 우변이 같은 기간을
본다. 달력 월 같은 근사를 쓰면 회귀가 서로 다른 기간을 비교하게 된다.

In [2]:
rows = []
for window in (30, 60, 120, 252):
    clustered = pd.read_parquet(PROCESSED_DIR / f'clustered_{window}df.parquet')
    perf = add_forward_returns(clustered, final_df)
    period_factors = compound_factors_to_periods(perf, ff)
    summary = summarize_by_group(perf, verbose=False)
    joined = summary.join(period_factors, how='inner')

    cols = ['Mkt-RF', 'SMB', 'HML', 'MOM', 'RF']
    rows.append(factor_regression(joined['stable_mean_return'], joined[cols],
                                  f'{window}일 · 안정군', subtract_rf=True))
    rows.append(factor_regression(joined['stable_minus_other'], joined[cols],
                                  f'{window}일 · 롱숏', subtract_rf=False))

attribution = pd.DataFrame(rows).set_index('label')
pd.set_option('display.width', 220)
attribution[['n', 'alpha', 'alpha_t', 'alpha_significant', 'adj_r2']].round(4)

[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)


[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)


[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)


[캐시] 편입/편출 이력 1259건 재사용 (/Users/choedasom/lab_middle_project/data/raw/cache/sp500_membership_history.csv)


,n,alpha,alpha_t,alpha_significant,adj_r2
label,,,,,
30일 · 안정군,85,0.0021,0.9108,False,0.8231
30일 · 롱숏,85,0.0019,0.5071,False,0.5143
60일 · 안정군,42,0.0036,0.6477,False,0.7498
60일 · 롱숏,42,-0.0011,-0.1613,False,0.6693
120일 · 안정군,20,-0.0136,-0.9271,False,0.7066
120일 · 롱숏,20,-0.0054,-0.3016,False,0.7687
252일 · 안정군,9,NaN,NaN,NaN,NaN
252일 · 롱숏,9,NaN,NaN,NaN,NaN


## 팩터 노출

In [3]:
attribution[['b_Mkt-RF', 't_Mkt-RF', 'b_SMB', 'b_HML', 'b_MOM', 'adj_r2']].round(3)

,b_Mkt-RF,t_Mkt-RF,b_SMB,b_HML,b_MOM,adj_r2
label,,,,,,
30일 · 안정군,0.754,16.367,-0.086,0.114,-0.206,0.823
30일 · 롱숏,-0.501,-6.466,-0.348,0.025,0.127,0.514
60일 · 안정군,0.705,8.412,0.065,0.171,-0.060,0.750
60일 · 롱숏,-0.399,-3.932,-0.563,-0.133,0.139,0.669
120일 · 안정군,0.799,5.362,-0.314,0.091,-0.202,0.707
120일 · 롱숏,-0.434,-2.376,-0.556,-0.349,0.147,0.769
252일 · 안정군,NaN,NaN,NaN,NaN,NaN,NaN
252일 · 롱숏,NaN,NaN,NaN,NaN,NaN,NaN


## 결과 해석

### 1. 알파는 어디에서도 유의하지 않다

알파의 t값이 **-0.93 ~ +0.91** 범위로, 어느 것도 |t| ≥ 2에 미치지 못한다. 안정군 포트폴리오에도,
롱숏 스프레드에도 **네 팩터가 설명하지 못하고 남는 수익이 없다.**

### 2. 대신 노출 구조가 아주 선명하게 나온다

- **안정군의 시장 베타 0.70 ~ 0.80** (t = 16.4 / 8.4 / 5.4로 매우 유의) — 저변동성 포트폴리오가
  당연히 가져야 할 방어적 노출이다. 설계 의도대로 작동했다는 뜻이기도 하다.
- **롱숏 스프레드의 시장 베타가 −0.40 ~ −0.50** (유의) — "안정군을 사고 나머지를 파는" 베팅은
  본질적으로 **시장 베타를 파는 포지션**이다.
- **SMB 계수가 음수** — 안정군이 대형주 쪽으로 기울어 있다. 대형주의 변동성이 낮으니 당연하다.
- **조정 R²가 0.51 ~ 0.82** — 전략 수익률 변동의 대부분이 이미 알려진 네 팩터로 설명된다.

### 3. 06번에서 발견했던 "시장 국면 의존성"의 정체가 여기서 밝혀진다

06·07번에서 안정-나머지 격차가 시장수익률과 **-0.64 ~ -0.72의 상관**을 보인다는 것을 관찰하고
"시장 국면에 좌우된다"고 서술했다. 그 현상의 **메커니즘이 바로 이 −0.40 ~ −0.50의 시장 베타**다.
상관관계로 관찰했던 것을 회귀계수로 정량화한 셈이다.

### 4. 그래서 정확한 결론

> 이 전략은 새로운 수익원을 찾지 못했다. **알려진 팩터 노출(낮은 시장 베타 + 대형주 편향)을
> 사는 하나의 방법**이고, 실제로 그렇게 작동했다. 위험대비수익이 높아 보였던 것도 알파가 아니라
> 이 노출 구조의 결과다.

이건 실패가 아니라 **정확한 자기 위치 파악**이다. 학술적으로도 저변동성 이상현상(low-volatility
anomaly)은 시장 베타를 파는 포지션(betting-against-beta)으로 상당 부분 설명된다고 알려져 있고,
이 결과는 그 문헌과 일치한다.

### 5. 252일 구간은 회귀 자체를 거부했다

관측 9개로 파라미터 5개를 추정할 수 없다. `factor_regression()`이 숫자를 내놓는 대신
`note`를 반환하도록 만들어, **답처럼 보이는 값을 출력하지 않게** 했다.

## 한계

- **팩터 데이터는 미국 전체 시장 기준**이고 우리 유니버스는 S&P 500이다. 완전히 같은 모집단이 아니다.
- **관측 수가 여전히 작다** — 30일 구간도 85개뿐이라, 알파가 없다기보다 **작은 알파를 검출할
  검정력이 없다**고 읽는 게 정확하다.
- **팩터 4개만 사용**했다. 품질(RMW)·투자(CMA)나 BAB 팩터를 직접 넣으면 설명력이 더 올라갈 수 있다.